In [2]:

import pandas as pd
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)


In [3]:

import os
from config.constants import DATA_PATH 
from dotenv import load_dotenv


load_dotenv(override=True)

GITHUB_TOKEN = os.getenv("GITHUB_TOKEN")

if GITHUB_TOKEN is None:
    raise ValueError(
        "GITHUB_TOKEN not found in .env file"
    )


HEADERS = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
}

df=pd.read_csv(f'{DATA_PATH}df_n0.csv')
df["patch_id"] = [f"P{i:06d}" for i in range(len(df))]


In [ ]:
import requests
import pandas as pd
import re

pr_cache = {}
files_cache = {}

def github_get(url):
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )
    if response.status_code != 200:
        print(
            f"GitHub error {response.status_code}: {response.text}"
        )
        return None
    return response.json()


def get_pr_metadata(repo, pr_number):
    key = (repo, pr_number)
    if key not in pr_cache:
        url = (
            f"https://api.github.com/repos/"
            f"{repo}/pulls/{pr_number}"
        )
        pr_cache[key] = github_get(url)
    return pr_cache[key]



def get_pr_files(repo, pr_number):
    key = (repo, pr_number)
    if key not in files_cache:
        url = (
            f"https://api.github.com/repos/"
            f"{repo}/pulls/{pr_number}/files"
        )
        files_cache[key] = github_get(url)
    return files_cache[key]


def split_patch_into_hunks(patch):
    if not patch:
        return []
    hunks = re.split(
        r'(?=^@@)',
        patch,
        flags=re.MULTILINE
    )
    return [
        hunk.strip()
        for hunk in hunks
        if hunk.strip()
    ]

def find_target_file(dataset_hunk, files):
    if not files:
        return None
    hunk_lines = []
    for line in dataset_hunk.splitlines():
        if line.startswith("@@"):
            continue
        if not line.strip():
            continue
        cleaned = line.lstrip("+- ").rstrip()
        if cleaned:
            hunk_lines.append(cleaned)

    best_file = None
    best_score = 0

    for file in files:
        patch = file.get("patch", "")
        score = sum(
            line in patch
            for line in hunk_lines
        )
        if score > best_score:
            best_score = score
            best_file = file["filename"]
    return best_file

processed_rows = 0
total_rows = len(df)

def enrich_github_information(row):
    global processed_rows
    repo = row["repo"]
    pr_number = int(row["ghid"])
    files = get_pr_files(
        repo,
        pr_number
    )
    pr = get_pr_metadata(
        repo,
        pr_number
    )

    if files is None:
        result = pd.Series({
            "pr_number": pr_number,
            "pr_title": (
                pr["title"]
                if pr
                else None
            ),
            "changed_files": [],
            "target_file": None,
            "num_code_hunks": 0,
            "pr_code_hunks": [],
            "same_file_code_hunks": [],
            "code_hunks_num_same_file": 0
        })

    else:
        # All changed files
        changed_files = [
            f["filename"]
            for f in files
        ]
        # if the PR changes only one file then all the code hunks would already exist in the target file code hunks
        if len(changed_files) == 1:
            pr_code_hunks = []
        else:
            pr_code_hunks = [
                {
                    "filename": f["filename"],
                    "patch": f["patch"]
                }
                for f in files
                if f.get("patch")
            ]


        target_file = find_target_file(
            row["hunk"],
            files
        )

        same_file_code_hunks = []

        if target_file:

            for f in files:

                if (
                    f.get("patch")
                    and f["filename"] == target_file
                ):

                    individual_hunks = split_patch_into_hunks(
                        f["patch"]
                    )

                    for hunk in individual_hunks:

                        same_file_code_hunks.append({
                            "filename": f["filename"],
                            "patch": hunk
                        })

        # Number of individual hunks in target file
        code_hunks_num_same_file = len(
            same_file_code_hunks
        )

        num_code_hunks = sum(
            len(split_patch_into_hunks(f["patch"]))
            for f in files
            if f.get("patch")
        )

        result = pd.Series({
            "pr_number": pr_number,
            "pr_title": (
                pr["title"]
                if pr
                else None
            ),

            "changed_files": changed_files,
            "target_file": target_file,
            "num_code_hunks": num_code_hunks,
            "pr_code_hunks": pr_code_hunks,
            "same_file_code_hunks": same_file_code_hunks,
            "code_hunks_num_same_file": code_hunks_num_same_file
        })

    processed_rows += 1
    if processed_rows % 20 == 0 or processed_rows == total_rows:
        print(
            f"Processed {processed_rows}/{total_rows}"
        )
    return result



In [ ]:
result_df = df.apply(
    enrich_github_information,
    axis=1
)

df_enriched = pd.concat(
    [df, result_df],
    axis=1
)

Processed 560/507
GitHub error 404: {"message":"Not Found","documentation_url":"https://docs.github.com/rest/pulls/pulls#list-pull-requests-files","status":"404"}
GitHub error 404: {"message":"Not Found","documentation_url":"https://docs.github.com/rest/pulls/pulls#get-a-pull-request","status":"404"}
Processed 580/507


In [ ]:
df_enriched.to_csv(f'{DATA_PATH}df_n1.csv',index=False)